# LeetCode 1085 — Sum of Digits in the Minimum Number bằng Python

**Link đề:** https://leetcode.com/problems/sum-of-digits-in-the-minimum-number/
**Độ khó:** Easy · **Chủ đề:** Array, Math

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def sumOfDigits(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Đọc kỹ đề có **nhiều bước nhỏ**: tìm số nhỏ nhất → tính tổng chữ số → xét chẵn/lẻ → trả `0`/`1` (chú ý: **lẻ → 0**, **chẵn → 1**).
2. Tự viết hàm tìm **min** bằng vòng lặp, rồi dùng `min()` có sẵn.
3. Tính tổng chữ số theo 2 cách: chuỗi (`str`, `int`) và toán (`% 10`, `// 10`); thêm `divmod`.
4. Kiểm tra chẵn/lẻ bằng `% 2` và bằng bit `& 1`; chuyển bool sang int.
5. Trace, chứng minh, Time/Space, lỗi hay gặp và bài tập mở rộng.

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài

Cho mảng số nguyên dương `nums`. Gọi `S` là **tổng các chữ số** của **phần tử nhỏ nhất** trong `nums`.

- Nếu `S` **lẻ** → trả về `0`.
- Nếu `S` **chẵn** → trả về `1`.

| `nums` | min | Tổng chữ số `S` | Output |
|---|:---:|:---:|:---:|
| `[34,23,1,24,75,33,54,8]` | 1 | 1 (lẻ) | `0` |
| `[99,77,33,66,55]` | 33 | 3 + 3 = 6 (chẵn) | `1` |

**Constraints (theo hiểu biết):**
- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`

Ba bước tách bạch: **(1) min**, **(2) tổng chữ số**, **(3) chẵn/lẻ → 0/1**. Ta học từng bước rồi ghép lại.

In [1]:
nums = [99, 77, 33, 66, 55]
m = min(nums)
s = sum(int(ch) for ch in str(m))
print("min =", m, "| tổng chữ số =", s, "| chẵn?", s % 2 == 0, "| đáp án =", 1 if s % 2 == 0 else 0)

min = 33 | tổng chữ số = 6 | chẵn? True | đáp án = 1


## 2. Bước 1 — Tìm phần tử nhỏ nhất

### 2.1. Tự viết bằng vòng lặp

Ý tưởng "nhà vô địch hiện tại": giữ `smallest` = nhỏ nhất **trong phần đã xem**. Gặp số nhỏ hơn thì thay.

Khởi tạo bằng `nums[0]` (đề đảm bảo mảng không rỗng), **không** khởi tạo bằng `0` — vì mọi số dương đều lớn hơn 0, `smallest` sẽ kẹt ở 0.

In [2]:
def find_min(nums: list[int]) -> int:
    smallest = nums[0]
    for x in nums[1:]:
        if x < smallest:
            smallest = x
    return smallest

print(find_min([34, 23, 1, 24, 75, 33, 54, 8]))   # 1
print(find_min([99, 77, 33, 66, 55]))             # 33

1
33


In [3]:
def trace_find_min(nums: list[int]) -> int:
    smallest = nums[0]
    print(f"bắt đầu: smallest = {smallest}")
    for x in nums[1:]:
        if x < smallest:
            print(f"  x={x:>3} < {smallest:>3} -> cập nhật smallest = {x}")
            smallest = x
        else:
            print(f"  x={x:>3} >= {smallest:>3} -> giữ nguyên")
    return smallest

trace_find_min([99, 77, 33, 66, 55])

bắt đầu: smallest = 99
  x= 77 <  99 -> cập nhật smallest = 77
  x= 33 <  77 -> cập nhật smallest = 33
  x= 66 >=  33 -> giữ nguyên
  x= 55 >=  33 -> giữ nguyên


33

### 2.2. Dùng `min()` có sẵn

`min(iterable)` trả phần tử nhỏ nhất, `O(n)`. Gọi `min([])` (rỗng) sẽ báo `ValueError` — nhưng đề đảm bảo có ít nhất 1 phần tử.

In [4]:
print(min([34, 23, 1, 24, 75, 33, 54, 8]), min([99, 77, 33, 66, 55]))
try:
    min([])
except ValueError as e:
    print("ValueError:", e)

1 33
ValueError: min() iterable argument is empty


## 3. Bước 2 — Tổng các chữ số

### 3.1. Cách chuỗi: `str(m)` rồi cộng từng ký tự

- `str(33)` → `"33"`; duyệt từng ký tự `"3"`, `"3"`.
- `int("3")` → `3` (đổi ký tự chữ số thành số).
- `sum(...)` cộng lại.

In [5]:
m = 75
print(list(str(m)))                       # ['7', '5']
print([int(ch) for ch in str(m)])         # [7, 5]
print(sum(int(ch) for ch in str(m)))      # 12

['7', '5']
[7, 5]
12


### 3.2. Cách toán: `% 10` lấy chữ số cuối, `// 10` bỏ chữ số cuối

```text
m = 75: 75 % 10 = 5, 75 // 10 = 7
m = 7 :  7 % 10 = 7,  7 // 10 = 0  -> dừng
tổng = 5 + 7 = 12
```

In [6]:
def digit_sum_math(m: int) -> int:
    total = 0
    while m > 0:
        total += m % 10     # cộng chữ số cuối
        m //= 10            # bỏ chữ số cuối
    return total

def trace_digit_sum(m: int) -> int:
    total = 0
    while m > 0:
        d = m % 10
        total += d
        print(f"m={m:>3}: chữ số cuối={d}, total={total}, m mới={m // 10}")
        m //= 10
    return total

print(digit_sum_math(75), digit_sum_math(100), digit_sum_math(1))
trace_digit_sum(100)

12 1 1
m=100: chữ số cuối=0, total=0, m mới=10
m= 10: chữ số cuối=0, total=0, m mới=1
m=  1: chữ số cuối=1, total=1, m mới=0


1

### 3.3. `divmod(a, b)` — lấy cả thương và dư một lần

`divmod(75, 10)` → `(7, 5)` = `(75 // 10, 75 % 10)`. Viết gọn: `m, d = divmod(m, 10)`.

In [7]:
print(divmod(75, 10))
m, total = 99, 0
while m:
    m, d = divmod(m, 10)
    total += d
print("tổng chữ số 99 =", total)

(7, 5)
tổng chữ số 99 = 18


## 4. Bước 3 — Chẵn/lẻ và trả về `0`/`1`

- `S % 2 == 0` ⇔ `S` chẵn. `S % 2` là `0` (chẵn) hoặc `1` (lẻ).
- Đề muốn **chẵn → 1, lẻ → 0** — **ngược** với `S % 2`! Nên đáp án = `1 - S % 2`.
- Cách khác: `int(S % 2 == 0)`: bool `True` → `1`, `False` → `0`.
- Cách bit: số lẻ có bit cuối là 1 → `S & 1` giống `S % 2` với số không âm.

In [8]:
for S in [1, 6, 12, 0]:
    print(f"S={S:>2}: S%2={S % 2}, S&1={S & 1}, 1-S%2={1 - S % 2}, int(S%2==0)={int(S % 2 == 0)}")

S= 1: S%2=1, S&1=1, 1-S%2=0, int(S%2==0)=0
S= 6: S%2=0, S&1=0, 1-S%2=1, int(S%2==0)=1
S=12: S%2=0, S&1=0, 1-S%2=1, int(S%2==0)=1
S= 0: S%2=0, S&1=0, 1-S%2=1, int(S%2==0)=1


### Vì sao `S & 1` cho biết chẵn/lẻ?

Viết `S` dạng nhị phân: bit cuối có giá trị 1, các bit khác đều là bội của 2. Vậy `S` lẻ ⇔ bit cuối = 1. Toán tử `&` (AND từng bit) với `1` giữ lại đúng bit cuối.

In [9]:
for S in [5, 6]:
    print(S, "nhị phân:", bin(S), "| S & 1 =", S & 1)

5 nhị phân: 0b101 | S & 1 = 1
6 nhị phân: 0b110 | S & 1 = 0


## 5. Cách 1 — Ghép 3 bước bằng chuỗi (dễ hiểu nhất)

In [10]:
def sum_of_digits_string(nums: list[int]) -> int:
    m = min(nums)                              # bước 1
    s = sum(int(ch) for ch in str(m))          # bước 2
    return 1 if s % 2 == 0 else 0              # bước 3

print(sum_of_digits_string([34, 23, 1, 24, 75, 33, 54, 8]))   # 0
print(sum_of_digits_string([99, 77, 33, 66, 55]))             # 1

0
1


### Biểu thức điều kiện `A if cond else B`

`1 if s % 2 == 0 else 0` đọc là: "nếu `s` chẵn thì 1, ngược lại 0". Đây là **biểu thức** (có giá trị), dùng được ngay sau `return`.

## 6. Cách 2 (lời giải chính) — Thuần toán, không đổi sang chuỗi

Ghép từng đoạn:

**Đoạn 1:** `m = min(nums)` (hoặc vòng lặp tự viết).

**Đoạn 2:** `while m > 0: total += m % 10; m //= 10`.

**Đoạn 3:** `return 1 - total % 2`.

In [11]:
def sum_of_digits_math(nums: list[int]) -> int:
    m = min(nums)                 # Đoạn 1
    total = 0
    while m > 0:                  # Đoạn 2
        total += m % 10
        m //= 10
    return 1 - total % 2          # Đoạn 3: chẵn -> 1, lẻ -> 0

print(sum_of_digits_math([34, 23, 1, 24, 75, 33, 54, 8]))
print(sum_of_digits_math([99, 77, 33, 66, 55]))

0
1


## 7. Trace toàn bộ với `nums = [99, 77, 33, 66, 55]`

| Giai đoạn | Biến | Giá trị |
|---|---|---|
| Tìm min | duyệt 99 → 77 → 33 → 66 → 55 | `m = 33` |
| Tổng chữ số | `33 % 10 = 3`, `m = 3`; `3 % 10 = 3`, `m = 0` | `total = 6` |
| Chẵn/lẻ | `6 % 2 = 0` | chẵn |
| Trả về | `1 - 0` | **`1`** |

In [12]:
def trace_sum_of_digits(nums: list[int]) -> int:
    print("nums =", nums)
    m = nums[0]
    for x in nums[1:]:
        if x < m:
            m = x
    print("min =", m)
    total, k = 0, m
    while k > 0:
        total += k % 10
        print(f"  cộng chữ số {k % 10} -> total = {total}")
        k //= 10
    ans = 1 - total % 2
    print(f"total = {total} ({'chẵn' if total % 2 == 0 else 'lẻ'}) -> trả {ans}\n")
    return ans

trace_sum_of_digits([99, 77, 33, 66, 55])
trace_sum_of_digits([34, 23, 1, 24, 75, 33, 54, 8])

nums = [99, 77, 33, 66, 55]
min = 33
  cộng chữ số 3 -> total = 3
  cộng chữ số 3 -> total = 6
total = 6 (chẵn) -> trả 1

nums = [34, 23, 1, 24, 75, 33, 54, 8]
min = 1
  cộng chữ số 1 -> total = 1
total = 1 (lẻ) -> trả 0



0

## 8. Cách 3 — Một lượt duy nhất, không gọi `min` riêng

Có thể làm gọn trong một vòng `for`: vừa tìm min. Sau đó tính tổng chữ số chỉ cho min. (Lưu ý: **không** nên tính tổng chữ số cho mọi phần tử rồi lấy của phần tử nhỏ nhất — thừa việc.)

Ở đây chủ yếu để thấy: `min` có tham số `key` — nhưng **cẩn thận**: `min(nums, key=digit_sum)` là "phần tử có tổng chữ số nhỏ nhất", **khác** với "tổng chữ số của phần tử nhỏ nhất"!

In [13]:
def digit_sum(x: int) -> int:
    return sum(int(c) for c in str(x))

nums = [19, 20]
print("min(nums) =", min(nums), "-> tổng chữ số", digit_sum(min(nums)))                      # 19 -> 10
print("min theo key=digit_sum =", min(nums, key=digit_sum), "(KHÔNG phải điều đề hỏi)")   # 20

def sum_of_digits_one_pass(nums: list[int]) -> int:
    smallest = nums[0]
    for x in nums:
        if x < smallest:
            smallest = x
    return int(digit_sum(smallest) % 2 == 0)

print(sum_of_digits_one_pass([99, 77, 33, 66, 55]))

min(nums) = 19 -> tổng chữ số 10
min theo key=digit_sum = 20 (KHÔNG phải điều đề hỏi)
1


### 8.1. Kỹ thuật "chia bước thành hàm nhỏ" và test từng bước

Bài nhiều bước rất dễ sai ở **một** bước mà không biết bước nào. Mẹo: viết mỗi bước thành một hàm có tên rõ nghĩa, test riêng từng hàm bằng `assert`, rồi mới ghép.

| Bước | Hàm | Input → Output mẫu |
|---|---|---|
| 1 | `step_min(nums)` | `[99, 77, 33]` → `33` |
| 2 | `step_digit_sum(m)` | `33` → `6`, `100` → `1` |
| 3 | `step_answer(s)` | `6` → `1`, `11` → `0` |

Khi một test tổng thể hỏng, chạy lại test của từng bước để khoanh vùng lỗi.

In [14]:
def step_min(nums):
    return min(nums)

def step_digit_sum(m):
    return digit_sum_math(m)

def step_answer(s):
    return 1 - s % 2

# test từng bước
assert step_min([99, 77, 33]) == 33
assert step_digit_sum(33) == 6 and step_digit_sum(100) == 1
assert step_answer(6) == 1 and step_answer(11) == 0
print("3 bước riêng lẻ đều PASS")

def sum_of_digits_pipeline(nums):
    return step_answer(step_digit_sum(step_min(nums)))

print(sum_of_digits_pipeline([99, 77, 33, 66, 55]), sum_of_digits_pipeline([34, 23, 1, 24, 75, 33, 54, 8]))

3 bước riêng lẻ đều PASS
1 0


### 8.2. Bảng so sánh các cách

| Cách | Ý tưởng | Ưu điểm | Nhược điểm |
|---|---|---|---|
| 1. Chuỗi | `sum(int(c) for c in str(min(nums)))` | ngắn, dễ đọc | tạo chuỗi tạm |
| 2. Toán | `% 10`, `// 10` | không cấp phát, kỹ năng nền | dài hơn vài dòng |
| 3. Một lượt tự viết | tự tìm min | không phụ thuộc hàm có sẵn | dễ sai khởi tạo |
| Pipeline | 3 hàm nhỏ | dễ test, dễ sửa | nhiều hàm cho bài nhỏ |

## 9. Code nộp LeetCode

### Bản toán học (lời giải chính)

In [15]:
class Solution:
    def sumOfDigits(self, nums: list[int]) -> int:
        m = min(nums)
        total = 0
        while m > 0:
            total += m % 10
            m //= 10
        return 1 - total % 2

sol = Solution()
assert sol.sumOfDigits([34, 23, 1, 24, 75, 33, 54, 8]) == 0
assert sol.sumOfDigits([99, 77, 33, 66, 55]) == 1
print("Solution PASS")

Solution PASS


### Bản chuỗi một dòng

In [16]:
class SolutionString:
    def sumOfDigits(self, nums: list[int]) -> int:
        return 1 - sum(map(int, str(min(nums)))) % 2

assert SolutionString().sumOfDigits([34, 23, 1, 24, 75, 33, 54, 8]) == 0
assert SolutionString().sumOfDigits([99, 77, 33, 66, 55]) == 1
print("SolutionString PASS")

SolutionString PASS


`map(int, "33")` áp `int` lên từng ký tự → sinh `3, 3`; `sum` cộng lại. Tương đương `sum(int(c) for c in "33")`.

## 10. Vì sao đúng?

- **Bước 1:** vòng tìm min giữ invariant "`smallest` là nhỏ nhất trong các phần tử đã duyệt"; sau khi duyệt hết thì là nhỏ nhất của cả mảng.
- **Bước 2:** với `m > 0` viết `m = 10·q + d` (`d = m % 10`, `q = m // 10`). Tổng chữ số của `m` = `d` + tổng chữ số của `q`. Vòng `while` áp dụng đúng công thức này cho tới khi `q = 0` ⇒ `total` đúng là tổng chữ số. (Với `nums[i] >= 1` nên `m >= 1`, không gặp ca `m = 0`.)
- **Bước 3:** `total % 2` bằng 0 khi chẵn, 1 khi lẻ ⇒ `1 - total % 2` bằng 1 khi chẵn, 0 khi lẻ — đúng như đề.

## 11. Độ phức tạp

Gọi `n = len(nums)`, `M = max giá trị` (≤ 100), số chữ số `d = O(log10 M)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. `min` + chuỗi | `O(n + d)` | `O(d)` (chuỗi số) |
| 2. `min` + `% 10` | `O(n + d)` | `O(1)` |
| 3. Một lượt tự viết | `O(n + d)` | `O(d)` |

Với `M <= 100`, `d <= 3` nên thực tế là `O(n)`.

## 12. Bộ test

Nhóm test:
1. Hai ví dụ đề.
2. Một phần tử: `[1]` → 0, `[2]` → 1.
3. Min là `100` (tổng 1, lẻ) → 0; min `10` (tổng 1) → 0; min `11` (tổng 2) → 1.
4. Min lặp lại nhiều lần.
5. Bẫy `key`: `[19, 20]` → min 19, tổng 10 chẵn → 1.
6. Đối chiếu cách trên toàn bộ mảng ngẫu nhiên.

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:22} PASS -> {actual}")

tests = [
    ("đề 1", [34, 23, 1, 24, 75, 33, 54, 8], 0),
    ("đề 2", [99, 77, 33, 66, 55], 1),
    ("[1]", [1], 0),
    ("[2]", [2], 1),
    ("min 100", [100], 0),
    ("min 10", [10, 50, 99], 0),
    ("min 11", [11, 12, 99], 1),
    ("min lặp", [8, 8, 8], 1),
    ("bẫy key [19, 20]", [19, 20], 1),
]
for name, nums, exp in tests:
    check(name, sum_of_digits_math(nums), exp)
    assert sum_of_digits_string(nums) == sum_of_digits_one_pass(nums) == exp

đề 1                   PASS -> 0
đề 2                   PASS -> 1
[1]                    PASS -> 0
[2]                    PASS -> 1
min 100                PASS -> 0
min 10                 PASS -> 0
min 11                 PASS -> 1
min lặp                PASS -> 1
bẫy key [19, 20]       PASS -> 1


In [18]:
import random
random.seed(1085)
for _ in range(5000):
    nums = [random.randint(1, 100) for _ in range(random.randint(1, 100))]
    r = {sum_of_digits_string(nums), sum_of_digits_math(nums), sum_of_digits_one_pass(nums),
         Solution().sumOfDigits(nums), SolutionString().sumOfDigits(nums)}
    assert len(r) == 1, nums
print("5000 test ngẫu nhiên: 5 cách khớp nhau")

5000 test ngẫu nhiên: 5 cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đảo ngược quy ước — trả `S % 2`

`S % 2` cho **lẻ → 1**, nhưng đề muốn **lẻ → 0**. Đây là lỗi phổ biến nhất của bài.

In [19]:
def wrong_parity(nums):
    return sum(int(c) for c in str(min(nums))) % 2

print("sai :", wrong_parity([99, 77, 33, 66, 55]), "| đúng:", sum_of_digits_math([99, 77, 33, 66, 55]))

sai : 0 | đúng: 1


### Lỗi 2: Khởi tạo `smallest = 0` khi tự tìm min

In [20]:
def wrong_min_init(nums):
    smallest = 0
    for x in nums:
        if x < smallest:
            smallest = x
    return smallest

print("sai :", wrong_min_init([99, 77, 33]), "| đúng:", min([99, 77, 33]))

sai : 0 | đúng: 33


**Sửa:** khởi tạo bằng `nums[0]` hoặc `float("inf")` (vô cực dương).

### Lỗi 3: Tính tổng chữ số của **tổng mảng** hoặc của **mọi phần tử**

Đề chỉ quan tâm **một số**: phần tử nhỏ nhất.

### Lỗi 4: Dùng `/ 10` thay vì `// 10`

`/` trả số thực: `33 / 10 = 3.3`, vòng `while m > 0` chạy rất nhiều lần với số thực ngày càng nhỏ, và `% 10` ra số thực → sai.

In [21]:
m = 33
print(m / 10, m // 10)

3.3 3


### Lỗi 5: Trả `True`/`False` thay vì `1`/`0`

Python coi `True == 1` nên đôi khi vẫn được chấp nhận, nhưng chữ ký đề yêu cầu `int`. Dùng `int(...)` hoặc `1 - S % 2` cho rõ ràng.

## 14. Mẹo nhận dạng pattern

- Đề "tách chữ số" → `% 10` / `// 10` (hoặc `str`). Đây là kỹ năng nền cho rất nhiều bài.
- Đề có **nhiều bước** → viết từng bước thành hàm nhỏ, test riêng, rồi ghép. Giảm lỗi đọc đề.
- Kiểm tra chẵn/lẻ → `% 2` hoặc `& 1`; nhớ đọc kỹ quy ước đầu ra.

| Bài liên quan | Ý tưởng |
|---|---|
| 258. Add Digits | cộng chữ số lặp đến 1 chữ số (digital root) |
| 1281. Subtract the Product and Sum of Digits | tích − tổng chữ số |
| 2160 / 1295. Find Numbers with Even Number of Digits | đếm chữ số, chẵn/lẻ |
| 2180. Count Integers With Even Digit Sum | tổng chữ số chẵn |

## 15. Biến thể

### Biến thể A — Digital root (cộng chữ số lặp lại tới khi còn 1 chữ số)

In [22]:
def digital_root(x: int) -> int:
    while x >= 10:
        x = digit_sum_math(x)
    return x

def digital_root_formula(x: int) -> int:
    return 0 if x == 0 else 1 + (x - 1) % 9      # công thức đồng dư mod 9

print(digital_root(38), digital_root_formula(38))   # 3 + 8 = 11 -> 1 + 1 = 2
assert all(digital_root(x) == digital_root_formula(x) for x in range(0, 5000))

2 2


### Biến thể B — Đếm số trong `[1, n]` có tổng chữ số chẵn (LeetCode 2180)

In [23]:
def count_even_digit_sum(n: int) -> int:
    return sum(1 for x in range(1, n + 1) if digit_sum_math(x) % 2 == 0)

print(count_even_digit_sum(4), count_even_digit_sum(30))   # 2 (2, 4), 14

2 14


## 16. Bài tập tự luyện (tăng dần)

**Bài 1 (tính tay):** `nums = [58, 47, 92, 64]`. Min là gì, tổng chữ số, và đáp án?

**Bài 2 (điền chỗ trống):** hoàn thiện `ex_digit_sum(x)` bằng `% 10`, `// 10`.

**Bài 3 (viết lại):** `ex_sum_of_digits(nums)` — không dùng `min()` có sẵn, không dùng `str`.

**Bài 4 (biến thể):** `ex_max_version(nums)` — cùng quy tắc nhưng áp dụng cho phần tử **lớn nhất**.

**Bài 5 (mini challenge):** `ex_min_by_digit_sum(nums)` — trả về phần tử có **tổng chữ số nhỏ nhất**; nếu hòa, chọn phần tử **nhỏ hơn**.

In [24]:
def ex_digit_sum(x: int) -> int:
    total = 0
    # TODO: while x > 0: cộng chữ số cuối, bỏ chữ số cuối
    return total

# assert ex_digit_sum(47) == 11 and ex_digit_sum(100) == 1

In [25]:
def ex_sum_of_digits(nums: list[int]) -> int:
    # TODO 1: tự tìm min (khởi tạo bằng nums[0])
    # TODO 2: tổng chữ số bằng toán
    # TODO 3: chẵn -> 1, lẻ -> 0
    pass

# assert ex_sum_of_digits([34, 23, 1, 24, 75, 33, 54, 8]) == 0
# assert ex_sum_of_digits([58, 47, 92, 64]) == 0

In [26]:
def ex_max_version(nums: list[int]) -> int:
    # TODO
    pass

# assert ex_max_version([58, 47, 92, 64]) == 0     # max 92 -> 11 lẻ
# assert ex_max_version([13, 2]) == 1              # max 13 -> 4 chẵn

In [27]:
def ex_min_by_digit_sum(nums: list[int]) -> int:
    # TODO: min với key là tuple (tổng chữ số, giá trị)
    pass

# assert ex_min_by_digit_sum([19, 20, 11]) == 11   # tổng chữ số: 10, 2, 2 -> hòa giữa 20 và 11 -> chọn 11

Bài 5 có bẫy **hòa**: `20` và `11` cùng tổng chữ số 2 → phải chọn số **nhỏ hơn** là `11`.

## 17. Gợi ý 3 mức

**Bài 1:** (1) Tìm min trước. (2) Min là `47`, tổng `4 + 7 = 11`. (3) `11` lẻ → `0`.

**Bài 2:** (1) Lấy chữ số cuối bằng `% 10`. (2) Bỏ chữ số cuối bằng `//= 10`. (3) `total += x % 10; x //= 10`.

**Bài 3:** (1) Ghép 3 bước. (2) `smallest = nums[0]` rồi duyệt. (3) `return 1 - total % 2`.

**Bài 4:** (1) Chỉ đổi bước 1. (2) Dùng `max` hoặc đảo dấu so sánh. (3) Phần còn lại giữ nguyên.

**Bài 5:** (1) Tiêu chí hai tầng: tổng chữ số rồi giá trị. (2) Python so tuple theo thứ tự từ điển. (3) `min(nums, key=lambda x: (digit_sum(x), x))`.

## 18. Đáp án tham khảo + template + checklist

Tự làm trước rồi mới xem. Bài 1: min `47`, tổng `11`, đáp án `0`.

In [28]:
def ans_digit_sum(x):
    total = 0
    while x > 0:
        total += x % 10
        x //= 10
    return total

def ans_sum_of_digits(nums):
    smallest = nums[0]
    for v in nums:
        if v < smallest:
            smallest = v
    return 1 - ans_digit_sum(smallest) % 2

def ans_max_version(nums):
    return 1 - ans_digit_sum(max(nums)) % 2

def ans_min_by_digit_sum(nums):
    return min(nums, key=lambda x: (ans_digit_sum(x), x))

assert ans_digit_sum(47) == 11 and ans_digit_sum(100) == 1
assert ans_sum_of_digits([34, 23, 1, 24, 75, 33, 54, 8]) == 0
assert ans_sum_of_digits([58, 47, 92, 64]) == 0
assert ans_max_version([58, 47, 92, 64]) == 0 and ans_max_version([13, 2]) == 1
assert ans_min_by_digit_sum([19, 20, 11]) == 11
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def sumOfDigits(self, nums: list[int]) -> int:
        m, total = min(nums), 0
        while m > 0:
            total += m % 10      # chữ số cuối
            m //= 10             # bỏ chữ số cuối
        return 1 - total % 2     # chẵn -> 1, lẻ -> 0
```

### Checklist trước khi submit
- [ ] Lấy **phần tử nhỏ nhất**, không phải tổng mảng hay phần tử có tổng chữ số nhỏ nhất.
- [ ] Dùng `// 10`, không `/ 10`.
- [ ] Quy ước đầu ra: **lẻ → 0, chẵn → 1** (ngược với `S % 2`).
- [ ] Trả `int`, không trả `bool`.
- [ ] Test `[1]` → 0, `[2]` → 1, `[100]` → 0.
- [ ] Time `O(n)`, Space `O(1)`.